# LeetCode 1497 — Check If Array Pairs Are Divisible by k bằng Python

Link đề: https://leetcode.com/problems/check-if-array-pairs-are-divisible-by-k/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def canArrange(self, arr: list[int], k: int) -> bool:
        ...
```

## 1. Mục tiêu học

1. Chuyển bài "ghép cặp có tổng chia hết cho `k`" sang bài **đếm số dư**: chỉ `x % k` của mỗi phần tử là quan trọng.
2. Nắm ba nhóm số dư đặc biệt: `r = 0` (tự ghép với nhau), `r = k/2` khi `k` chẵn (tự ghép với nhau), và cặp `r` ↔ `k − r`.
3. Hiểu cách Python xử lý `%` với **số âm** (luôn trả về `[0, k)`), khác C++/Java.
4. Viết 4 cách: quay lui (kiểm chứng), ghép tham lam một lượt bằng `dict`, đếm số dư `O(n + k)`, sắp xếp theo số dư + hai con trỏ.
5. Biết các bẫy: quên nhóm `k/2`, chỉ kiểm tra tổng, dùng phép dư kiểu C cho số âm.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` có độ dài **chẵn** `n` và số nguyên `k`. Ta muốn chia mảng thành đúng `n / 2` cặp sao cho **tổng mỗi cặp chia hết cho `k`**. Trả về `True` nếu làm được, ngược lại `False`.

**Ví dụ 1:** `arr = [1,2,3,4,5,10,6,7,8,9], k = 5` → `True` (các cặp `(1,9),(2,8),(3,7),(4,6),(5,10)`).

**Ví dụ 2:** `arr = [1,2,3,4,5,6], k = 7` → `True` (`(1,6),(2,5),(3,4)`).

**Ví dụ 3:** `arr = [1,2,3,4,5,6], k = 10` → `False`.

**Ràng buộc:** `n == len(arr)`, `1 <= n <= 10^5`, `n` chẵn; `-10^9 <= arr[i] <= 10^9`; `1 <= k <= 10^5`.

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [
    (([1, 2, 3, 4, 5, 10, 6, 7, 8, 9], 5), True),
    (([1, 2, 3, 4, 5, 6], 7), True),
    (([1, 2, 3, 4, 5, 6], 10), False),
]
for (arr, k), want in examples:
    print(f"arr={arr}, k={k} -> {want} | số dư: {[x % k for x in arr]}")

arr=[1, 2, 3, 4, 5, 10, 6, 7, 8, 9], k=5 -> True | số dư: [1, 2, 3, 4, 0, 0, 1, 2, 3, 4]
arr=[1, 2, 3, 4, 5, 6], k=7 -> True | số dư: [1, 2, 3, 4, 5, 6]
arr=[1, 2, 3, 4, 5, 6], k=10 -> False | số dư: [1, 2, 3, 4, 5, 6]


## 3. Kiến thức nền

### 3.1 Tính chất của phép chia lấy dư

`(a + b) % k == (a % k + b % k) % k`. Vậy `a + b` chia hết cho `k` ⇔ `(a % k + b % k) % k == 0` ⇔ hai số dư `r1, r2` thoả **`r1 + r2 == 0`** hoặc **`r1 + r2 == k`**.

In [2]:
k = 5
for a, b in [(1, 9), (7, 13), (5, 10), (2, 4)]:
    print(f"{a}+{b}={a+b}: chia hết? {(a + b) % k == 0} | dư {a % k} + {b % k}")

1+9=10: chia hết? True | dư 1 + 4
7+13=20: chia hết? True | dư 2 + 3
5+10=15: chia hết? True | dư 0 + 0
2+4=6: chia hết? False | dư 2 + 4


### 3.2 Số âm và `%` trong Python

Python định nghĩa `a % k` cùng dấu với `k` ⇒ với `k > 0` luôn được số trong `[0, k)`. C++/Java trả số âm khi `a < 0` ⇒ ở đó phải viết `((a % k) + k) % k`. Hàm `math.fmod` của Python thì cư xử như C.

In [3]:
import math
for a in [-7, -5, -1, 3]:
    print(f"a={a}: Python a % 5 = {a % 5} | kiểu C (math.fmod) = {int(math.fmod(a, 5))}")

a=-7: Python a % 5 = 3 | kiểu C (math.fmod) = -2
a=-5: Python a % 5 = 0 | kiểu C (math.fmod) = 0
a=-1: Python a % 5 = 4 | kiểu C (math.fmod) = -1
a=3: Python a % 5 = 3 | kiểu C (math.fmod) = 3


### 3.3 Đếm tần suất bằng `Counter` / mảng

`Counter(x % k for x in arr)` cho số lần xuất hiện mỗi số dư. Vì số dư nằm trong `[0, k)`, cũng có thể dùng mảng `cnt = [0] * k` (nhanh hơn khi `k` không quá lớn).

In [4]:
arr, k = [1, 2, 3, 4, 5, 10, 6, 7, 8, 9], 5
print(Counter(x % k for x in arr))
cnt = [0] * k
for x in arr:
    cnt[x % k] += 1
print(cnt)

Counter({1: 2, 2: 2, 3: 2, 4: 2, 0: 2})
[2, 2, 2, 2, 2]


### 3.4 Sắp xếp và hai con trỏ

Nếu sắp xếp các số dư khác 0, phần tử nhỏ nhất phải ghép với phần tử lớn nhất (nếu `r1 + r2 == k`). Hai con trỏ `i` từ trái, `j` từ phải kiểm tra từng cặp.

In [5]:
rems = sorted([1, 4, 2, 3, 3, 2])
i, j = 0, len(rems) - 1
while i < j:
    print(f"ghép {rems[i]} với {rems[j]}: tổng {rems[i] + rems[j]}")
    i += 1; j -= 1

ghép 1 với 4: tổng 5
ghép 2 với 3: tổng 5
ghép 2 với 3: tổng 5


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**, `k = 5`: số dư `[1,2,3,4,0,0,1,2,3,4]`.

| số dư r | cnt[r] | phải ghép với | cnt[đối tác] | OK? |
|---|---|---|---|---|
| 0 | 2 | chính nhóm 0 | — | chẵn ✓ |
| 1 | 2 | 4 | 2 | bằng nhau ✓ |
| 2 | 2 | 3 | 2 | bằng nhau ✓ |

**Ví dụ 3**, `k = 10`: số dư `[1,2,3,4,5,6]`. Nhóm 1 cần nhóm 9 (có 0 phần tử) ⇒ ✗ ⇒ `False`.

Quan sát: các phần tử **cùng số dư** hoàn toàn thay thế được cho nhau. Bài chỉ còn là: số lượng mỗi nhóm có "khớp" với nhóm đối tác không. Nhóm 0 và (khi `k` chẵn) nhóm `k/2` là nhóm **tự đối tác** ⇒ phải có số lượng chẵn.

In [6]:
for (arr, k), want in examples:
    cnt = Counter(x % k for x in arr)
    rows = [(r, cnt[r], (k - r) % k, cnt[(k - r) % k]) for r in sorted(cnt)]
    print(f"k={k}: (r, cnt[r], đối tác, cnt[đối tác]) = {rows} -> mong đợi {want}")

k=5: (r, cnt[r], đối tác, cnt[đối tác]) = [(0, 2, 0, 2), (1, 2, 4, 2), (2, 2, 3, 2), (3, 2, 2, 2), (4, 2, 1, 2)] -> mong đợi True
k=7: (r, cnt[r], đối tác, cnt[đối tác]) = [(1, 1, 6, 1), (2, 1, 5, 1), (3, 1, 4, 1), (4, 1, 3, 1), (5, 1, 2, 1), (6, 1, 1, 1)] -> mong đợi True
k=10: (r, cnt[r], đối tác, cnt[đối tác]) = [(1, 1, 9, 0), (2, 1, 8, 0), (3, 1, 7, 0), (4, 1, 6, 1), (5, 1, 5, 1), (6, 1, 4, 1)] -> mong đợi False


## 5. Cách 1 — Brute Force / baseline: quay lui ghép cặp

**Ý tưởng:** lấy phần tử chưa ghép đầu tiên, thử ghép với từng phần tử chưa ghép khác có tổng chia hết cho `k`, đệ quy phần còn lại. Nếu nhánh nào ghép hết ⇒ `True`.

In [7]:
def can_arrange_brute(arr, k):
    n = len(arr)
    used = [False] * n
    def go():
        i = next((t for t in range(n) if not used[t]), -1)
        if i == -1:
            return True
        used[i] = True
        for j in range(i + 1, n):
            if not used[j] and (arr[i] + arr[j]) % k == 0:
                used[j] = True
                if go():
                    return True
                used[j] = False
        used[i] = False
        return False
    return go()

for (arr, k), want in examples:
    print(can_arrange_brute(arr, k), want)

True True
True True
False False


**Độ phức tạp:** tệ nhất cỡ `(n−1)·(n−3)·…·1 = (n−1)!!` cách ghép ⇒ hàm mũ. Chỉ dùng với `n ≤ 12` để kiểm chứng.

**Nút thắt:** ta phân biệt từng phần tử, trong khi các phần tử cùng số dư là **như nhau** với bài toán. Chỉ cần biết **mỗi số dư xuất hiện bao nhiêu lần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** thay mỗi phần tử bằng `r = x % k`. Một cặp hợp lệ khi `r1 + r2 ∈ {0, k}`, tức là:
- `r1 = r2 = 0`; hoặc
- `r2 = k − r1` với `1 ≤ r1 ≤ k − 1` (khi `k` chẵn và `r1 = k/2` thì `r2 = r1`).

**Quy tắc (điều kiện cần và đủ):**
- `cnt[0]` chẵn;
- với mọi `1 ≤ r < k − r`: `cnt[r] == cnt[k − r]`;
- nếu `k` chẵn: `cnt[k/2]` chẵn.

**Nhận xét thú vị:** vì `n` chẵn, nếu hai điều kiện cuối đúng thì tổng các nhóm khác 0 là chẵn ⇒ `cnt[0]` tự động chẵn. Nhưng kiểm tra cả ba cho rõ ràng và an toàn.

**Cách nhìn "ghép tham lam":** duyệt từng phần tử; nếu đang có một phần tử "chờ" với số dư đối tác `(k − r) % k`, ghép ngay; nếu không, cho phần tử này vào hàng chờ. Vì phần tử cùng số dư thay thế được cho nhau, ghép sớm không bao giờ làm hỏng khả năng ghép sau này. Cuối cùng hàng chờ rỗng ⇔ `True`.

## 7. Các cách tiếp theo

### Cách 2 — Ghép tham lam một lượt với `dict` "đang chờ"

`(k − r) % k` xử lý gọn cả trường hợp `r = 0` (đối tác là 0).

In [8]:
def can_arrange_greedy(arr, k):
    waiting = {}                      # số dư -> số phần tử đang chờ ghép
    for x in arr:
        r = x % k
        need = (k - r) % k
        if waiting.get(need, 0) > 0:
            waiting[need] -= 1        # ghép với một phần tử đang chờ
        else:
            waiting[r] = waiting.get(r, 0) + 1
    return all(v == 0 for v in waiting.values())

class SolutionGreedy:
    def canArrange(self, arr: list[int], k: int) -> bool:
        return can_arrange_greedy(arr, k)

for (arr, k), want in examples:
    print(can_arrange_greedy(arr, k), want)

True True
True True
False False


**Độ phức tạp:** `O(n)` thời gian, `O(min(n, k))` bộ nhớ.

### Cách 3 — Đếm số dư và kiểm tra điều kiện (chuẩn)

1. `cnt[r]` cho `r ∈ [0, k)`.
2. Kiểm tra `cnt[0] % 2 == 0`.
3. Với `r` từ 1 tới `k − 1`: `cnt[r] == cnt[k − r]` (khi `r == k − r` điều kiện này luôn đúng, nên phải kiểm tra riêng tính chẵn).

In [9]:
def can_arrange_count(arr, k):
    cnt = [0] * k
    for x in arr:
        cnt[x % k] += 1
    if cnt[0] % 2:
        return False
    for r in range(1, k // 2 + 1):
        if r == k - r:
            if cnt[r] % 2:
                return False
        elif cnt[r] != cnt[k - r]:
            return False
    return True

for (arr, k), want in examples:
    print(can_arrange_count(arr, k), want)

True True
True True
False False


Chỉ cần duyệt `r` tới `k // 2` vì cặp `(r, k − r)` đối xứng. Code nộp LeetCode (Cách 3):

In [10]:
class SolutionCount:
    def canArrange(self, arr: list[int], k: int) -> bool:
        cnt = [0] * k
        for x in arr:
            cnt[x % k] += 1
        if cnt[0] % 2:
            return False
        for r in range(1, k // 2 + 1):
            if r == k - r:
                if cnt[r] % 2:
                    return False
            elif cnt[r] != cnt[k - r]:
                return False
        return True

print([SolutionCount().canArrange(a, k) for (a, k), _ in examples])

[True, True, False]


**Độ phức tạp:** `O(n + k)` thời gian, `O(k)` bộ nhớ. Với `k ≤ 10^5` rất nhanh.

### Cách 4 — Sắp xếp số dư + hai con trỏ

Tách số dư 0 (đếm chẵn), sắp xếp các số dư khác 0, rồi ghép **nhỏ nhất với lớn nhất**: mọi cặp phải có tổng đúng bằng `k`. Vì nếu phần tử nhỏ nhất `a` ghép được thì đối tác của nó là `k − a` — **lớn nhất** có thể; nếu phần tử lớn nhất hiện tại khác `k − a` thì hoặc nó không có đối tác, hoặc `a` không có đối tác.

In [11]:
def can_arrange_sort(arr, k):
    rems = sorted(x % k for x in arr)
    zeros = 0
    while zeros < len(rems) and rems[zeros] == 0:
        zeros += 1
    if zeros % 2:
        return False
    i, j = zeros, len(rems) - 1
    while i < j:
        if rems[i] + rems[j] != k:
            return False
        i += 1; j -= 1
    return True

class SolutionSort:
    def canArrange(self, arr: list[int], k: int) -> bool:
        return can_arrange_sort(arr, k)

for (arr, k), want in examples:
    print(can_arrange_sort(arr, k), want)

True True
True True
False False


**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ. Không phụ thuộc `k` — hữu ích nếu `k` rất lớn (ví dụ `10^18`) khiến mảng `cnt` không khả thi.

## 8. TRACE từng bước

Trace Cách 2 (ghép tham lam) trên `arr = [-1, 3, 7, -5, 9, 2, 6, -2], k = 4` (có số âm; kết quả kiểm chứng bằng quay lui).

| x | r = x % 4 | need | waiting trước | hành động |
|---|---|---|---|---|
| -1 | 3 | 1 | {} | chờ 3 |
| 3 | 3 | 1 | {3:1} | chờ 3 |
| 7 | 3 | 1 | {3:2} | chờ 3 |
| -5 | 3 | 1 | {3:3} | chờ 3 |
| 9 | 1 | 3 | {3:4} | ghép với một số dư 3 |
| 2 | 2 | 2 | {3:3} | chờ 2 |
| 6 | 2 | 2 | {3:3, 2:1} | ghép với số dư 2 |
| -2 | 2 | 2 | {3:3, 2:0} | chờ 2 |
| kết thúc | | | {3:3, 2:1} | còn chờ ⇒ **False** |

In [12]:
def trace_greedy(arr, k):
    waiting = {}
    for x in arr:
        r = x % k
        need = (k - r) % k
        if waiting.get(need, 0) > 0:
            waiting[need] -= 1
            act = f"ghép với một phần tử dư {need}"
        else:
            waiting[r] = waiting.get(r, 0) + 1
            act = f"cho vào hàng chờ dư {r}"
        print(f"x={x:3} r={r} need={need} -> {act:28} | waiting={waiting}")
    ok = all(v == 0 for v in waiting.values())
    print("=>", ok)
    return ok

arr_t, k_t = [-1, 3, 7, -5, 9, 2, 6, -2], 4
trace_greedy(arr_t, k_t)
print("quay lui xác nhận:", can_arrange_brute(arr_t, k_t))

x= -1 r=3 need=1 -> cho vào hàng chờ dư 3        | waiting={3: 1}
x=  3 r=3 need=1 -> cho vào hàng chờ dư 3        | waiting={3: 2}
x=  7 r=3 need=1 -> cho vào hàng chờ dư 3        | waiting={3: 3}
x= -5 r=3 need=1 -> cho vào hàng chờ dư 3        | waiting={3: 4}
x=  9 r=1 need=3 -> ghép với một phần tử dư 3    | waiting={3: 3}
x=  2 r=2 need=2 -> cho vào hàng chờ dư 2        | waiting={3: 3, 2: 1}
x=  6 r=2 need=2 -> ghép với một phần tử dư 2    | waiting={3: 3, 2: 0}
x= -2 r=2 need=2 -> cho vào hàng chờ dư 2        | waiting={3: 3, 2: 1}
=> False
quay lui xác nhận: False


Trace Cách 3 (đếm) trên Ví dụ 1 và trên một mảng `k` chẵn có nhóm `k/2`:

In [13]:
def trace_count(arr, k):
    cnt = [0] * k
    for x in arr:
        cnt[x % k] += 1
    print(f"k={k}, cnt={cnt}")
    print(f"  r=0: cnt={cnt[0]} -> {'chẵn' if cnt[0] % 2 == 0 else 'LẺ'}")
    if cnt[0] % 2:
        return False
    for r in range(1, k // 2 + 1):
        if r == k - r:
            print(f"  r={r} (= k/2): cnt={cnt[r]} -> {'chẵn' if cnt[r] % 2 == 0 else 'LẺ'}")
            if cnt[r] % 2:
                return False
        else:
            print(f"  r={r} vs {k - r}: {cnt[r]} vs {cnt[k - r]} -> {'khớp' if cnt[r] == cnt[k - r] else 'LỆCH'}")
            if cnt[r] != cnt[k - r]:
                return False
    return True

print(trace_count([1, 2, 3, 4, 5, 10, 6, 7, 8, 9], 5))
print(trace_count([2, 6, 1, 3, 10, 14], 4))

k=5, cnt=[2, 2, 2, 2, 2]
  r=0: cnt=2 -> chẵn
  r=1 vs 4: 2 vs 2 -> khớp
  r=2 vs 3: 2 vs 2 -> khớp
True
k=4, cnt=[0, 1, 4, 1]
  r=0: cnt=0 -> chẵn
  r=1 vs 3: 1 vs 1 -> khớp
  r=2 (= k/2): cnt=4 -> chẵn
True


## 9. Code nộp LeetCode

Bản chính: Cách 3 (đếm số dư).

In [14]:
class Solution:
    def canArrange(self, arr: list[int], k: int) -> bool:
        cnt = [0] * k
        for x in arr:
            cnt[x % k] += 1            # Python: luôn trong [0, k), kể cả x âm
        if cnt[0] % 2:
            return False
        for r in range(1, k // 2 + 1):
            if r == k - r:             # k chẵn, nhóm k/2 tự ghép với nhau
                if cnt[r] % 2:
                    return False
            elif cnt[r] != cnt[k - r]:
                return False
        return True

for (arr, k), want in examples:
    assert Solution().canArrange(arr, k) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Điều kiện cần.** Trong một cách ghép hợp lệ, mỗi phần tử dư 0 ghép với một phần tử dư 0 ⇒ `cnt[0]` chẵn. Phần tử dư `r` (`1 ≤ r ≤ k−1`) ghép với phần tử dư `k − r`. Nếu `r ≠ k − r` thì các cặp tạo song ánh giữa nhóm `r` và nhóm `k − r` ⇒ `cnt[r] = cnt[k − r]`. Nếu `r = k − r` (`k` chẵn, `r = k/2`) thì nhóm tự ghép ⇒ `cnt[r]` chẵn.

**Điều kiện đủ.** Nếu các điều kiện đúng, ta dựng cách ghép: ghép các phần tử dư 0 thành từng đôi; với mỗi `r < k − r`, ghép phần tử thứ `i` của nhóm `r` với phần tử thứ `i` của nhóm `k − r`; nhóm `k/2` ghép từng đôi. Mọi cặp đều có tổng số dư `0` hoặc `k` ⇒ tổng chia hết cho `k`. Mỗi phần tử được dùng đúng một lần.

**Cách 2 (tham lam).** Bất biến: `waiting` không bao giờ đồng thời có phần tử chờ ở hai nhóm đối tác `r` và `k − r` với `r ≠ k−r` (vì phần tử tới sau sẽ ghép ngay), và mỗi nhóm tự-đối-tác có tối đa 1 phần tử chờ. Cuối cùng `waiting` toàn 0 ⇔ các điều kiện trên đúng.

## 11. Phân tích Time / Space

Gọi `n = len(arr)` (`≤ 10^5`), `k ≤ 10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui | `O((n−1)!!)` tệ nhất | `O(n)` |
| 2. Ghép tham lam (dict) | `O(n)` | `O(min(n, k))` |
| 3. Đếm số dư | `O(n + k)` | `O(k)` |
| 4. Sort + hai con trỏ | `O(n log n)` | `O(n)` |

Phép `%` trên số nguyên tới `10^9` là `O(1)`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [15]:
ALL = {
    "brute": can_arrange_brute,
    "greedy": can_arrange_greedy,
    "count": can_arrange_count,
    "sort": can_arrange_sort,
    "sub_greedy": lambda a, k: SolutionGreedy().canArrange(a, k),
    "sub_count": lambda a, k: SolutionCount().canArrange(a, k),
    "sub_sort": lambda a, k: SolutionSort().canArrange(a, k),
    "submit": lambda a, k: Solution().canArrange(a, k),
}
edge = [
    (([5, 7], 1), True),                      # k = 1: mọi tổng đều chia hết
    (([-1, 1], 2), True),                     # số âm
    (([-10, 10], 2), True),
    (([-1, -1, -1, -1, 2, 2], 3), False),     # toàn số dư 2, không có số dư 1
    (([-1, 1, -2, 2, -3, 3, -4, 4], 3), True),
    (([2, 6, 1, 3], 4), True),                # nhóm k/2 = 2 có 2 phần tử
    (([2, 1, 3, 4], 4), False),               # nhóm k/2 lẻ, nhóm 0 lẻ
    (([0, 0], 7), True),
    (([10**9, -10**9], 10**5), True),         # giá trị cực
    (([1, 1, 1, 3], 3), False),               # tổng chia hết cho 3 nhưng không ghép được
]
for (arr, k), want in edge:
    assert can_arrange_brute(arr, k) == want, (arr, k)
for name, f in ALL.items():
    for (arr, k), want in examples + edge:
        assert f(arr, k) == want, (name, arr, k)
print("Mọi cách đúng với", len(examples + edge), "test (edge đã kiểm chứng bằng quay lui)")

Mọi cách đúng với 13 test (edge đã kiểm chứng bằng quay lui)


### 12.2 Ngẫu nhiên, đối chiếu quay lui (seed cố định)

Một nửa số test được **dựng để chắc chắn ghép được** (sinh `a`, rồi chọn `b ≡ −a (mod k)`), rồi xáo trộn và đôi khi sửa nhẹ một phần tử — để có đủ cả `True` lẫn `False`.

In [16]:
def random_case(rng):
    k = rng.randint(1, 10)
    m = rng.randint(1, 5)
    if rng.random() < 0.5:
        arr = []
        for _ in range(m):
            a = rng.randint(-30, 30)
            b = rng.randint(-5, 5) * k - a
            arr += [a, b]
        if rng.random() < 0.3:
            arr[rng.randrange(len(arr))] += rng.randint(1, 3)
        rng.shuffle(arr)
    else:
        arr = [rng.randint(-30, 30) for _ in range(2 * m)]
    return arr, k

rng = random.Random(1497)
stat = Counter()
for _ in range(3000):
    arr, k = random_case(rng)
    want = can_arrange_brute(arr, k)
    stat[want] += 1
    for name, f in ALL.items():
        assert f(arr, k) == want, (name, arr, k)
print("3000 test ngẫu nhiên khớp quay lui |", dict(stat))

3000 test ngẫu nhiên khớp quay lui | {True: 1405, False: 1595}


### 12.3 Giới hạn `n = 10^5`, `k = 10^5`

In [17]:
k_big = 10**5
half = [rng.randint(-10**9, 10**9) for _ in range(5 * 10**4)]
good = half + [(-x) % k_big + k_big * rng.randint(-1000, 1000) for x in half]
rng.shuffle(good)
bad = good[:]; bad[0] += 1
for label, arr in [("ghép được", good), ("lệch 1", bad)]:
    for name in ["greedy", "count", "sort", "submit"]:
        t0 = time.perf_counter(); r = ALL[name](arr, k_big); dt = time.perf_counter() - t0
        print(f"{label:9} | {name:7}: {r}  ({dt*1000:.1f} ms)")
assert Solution().canArrange(good, k_big) and not Solution().canArrange(bad, k_big)

ghép được | greedy : True  (31.3 ms)
ghép được | count  : True  (11.6 ms)
ghép được | sort   : True  (34.8 ms)
ghép được | submit : True  (13.7 ms)


lệch 1    | greedy : False  (27.1 ms)
lệch 1    | count  : False  (8.2 ms)
lệch 1    | sort   : False  (30.5 ms)
lệch 1    | submit : False  (11.5 ms)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng phép dư kiểu C với số âm

Trong C++/Java, `-1 % 2 == -1`. Mô phỏng bằng `math.fmod` trong Python: số dư âm không khớp với bảng đếm.

In [18]:
def wrong1(arr, k):
    cnt = Counter(int(math.fmod(x, k)) for x in arr)   # SAI: dư có thể âm
    if cnt[0] % 2:
        return False
    for r in range(1, k // 2 + 1):
        if r == k - r:
            if cnt[r] % 2:
                return False
        elif cnt[r] != cnt[k - r]:
            return False
    return True

for arr, k in [([-1, 1], 2), ([-1, 1, -2, 2, -3, 3, -4, 4], 3)]:
    print(arr, k, "| sai:", wrong1(arr, k), "| đúng:", Solution().canArrange(arr, k))

[-1, 1] 2 | sai: False | đúng: True
[-1, 1, -2, 2, -3, 3, -4, 4] 3 | sai: False | đúng: True


`-1 + 1 = 0` chia hết cho 2 nhưng bản sai trả `False`. **Sửa:** trong Python dùng `x % k`; trong C++/Java dùng `((x % k) + k) % k`.

### Lỗi 2 — Quên kiểm tra tính chẵn của nhóm `0` và nhóm `k/2`

Chỉ kiểm tra `cnt[r] == cnt[k − r]` cho mọi `r` từ 1 tới `k − 1`. Với `r = k/2`, điều kiện tự đúng (so một số với chính nó).

In [19]:
def wrong2(arr, k):
    cnt = [0] * k
    for x in arr:
        cnt[x % k] += 1
    return all(cnt[r] == cnt[k - r] for r in range(1, k))   # SAI: thiếu nhóm 0 và k/2

for arr, k in [([2, 1, 3, 4], 4), ([2, 6, 10, 1, 3, 4], 4)]:
    print(arr, k, "| sai:", wrong2(arr, k), "| đúng:", Solution().canArrange(arr, k))

[2, 1, 3, 4] 4 | sai: True | đúng: False
[2, 6, 10, 1, 3, 4] 4 | sai: True | đúng: False


`[2,1,3,4], k=4`: số dư `2,1,3,0` — nhóm 2 lẻ, nhóm 0 lẻ ⇒ không ghép được, bản sai vẫn trả `True`.

### Lỗi 3 — Chỉ kiểm tra tổng cả mảng chia hết cho `k`

Tổng các cặp chia hết ⇒ tổng cả mảng chia hết, nhưng **ngược lại không đúng**.

In [20]:
def wrong3(arr, k):
    return sum(arr) % k == 0                                # SAI: chỉ là điều kiện cần

for arr, k in [([1, 1, 1, 3], 3), ([1, 2, 3, 4, 5, 6, 7, 2], 10)]:
    print(arr, k, "tổng", sum(arr), "| sai:", wrong3(arr, k), "| đúng:", Solution().canArrange(arr, k))

[1, 1, 1, 3] 3 tổng 6 | sai: True | đúng: False
[1, 2, 3, 4, 5, 6, 7, 2] 10 tổng 30 | sai: True | đúng: False


### Lỗi 4 — Truy cập `cnt[k − r]` với `r = 0`

Viết vòng `for r in range(k)` và so `cnt[r]` với `cnt[k − r]` ⇒ khi `r = 0` truy cập `cnt[k]` — ngoài mảng (Python báo `IndexError`; ta bắt lỗi để minh hoạ).

In [21]:
def wrong4(arr, k):
    cnt = [0] * k
    for x in arr:
        cnt[x % k] += 1
    for r in range(k):
        if cnt[r] != cnt[k - r]:                             # SAI khi r = 0
            return False
    return True

try:
    print("sai:", wrong4([1, 4], 5))
except IndexError as e:
    print("sai: IndexError ->", e)
print("đúng:", Solution().canArrange([1, 4], 5))

sai: IndexError -> list index out of range
đúng: True


**Sửa:** dùng `(k − r) % k`, hoặc xử lý `r = 0` riêng và chỉ duyệt `r` từ 1 tới `k // 2`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng/hiệu chia hết cho `k`" ⇒ thay số bằng **số dư**; phần tử cùng số dư là **tương đương**.
- Ghép cặp `r` ↔ `k − r`; nhóm **tự đối tác**: `0` và `k/2` (khi `k` chẵn) ⇒ cần chẵn.
- Viết `need = (k - r) % k` để xử lý gọn `r = 0`.
- Python `%` với `k > 0` luôn không âm — nhưng nhớ cách sửa khi viết ngôn ngữ khác.
- Test: `k = 1`, `k = 2`, số âm, nhóm `k/2` lẻ.

## 15. Biến thể của bài

1. **LC 1010 — Pairs of Songs With Total Durations Divisible by 60:** **đếm số cặp** `(i < j)` có tổng chia hết ⇒ cũng dùng `cnt` số dư (Bài 4).
2. **LC 2183 / 2364:** đếm cặp theo điều kiện chia hết khác (tích, hiệu).
3. **Hiệu chia hết cho `k`:** `a − b ≡ 0` ⇔ cùng số dư ⇒ mỗi nhóm số dư phải chẵn.
4. **Bỏ ít phần tử nhất để ghép được** (Bài 5).
5. **Ghép thành nhóm 3 phần tử có tổng chia hết cho `k`:** khó hơn nhiều — không còn là ghép đôi đơn giản.

In [22]:
# Minh hoạ biến thể 3: chia thành cặp có HIỆU chia hết cho k
def can_pair_diff(arr, k):
    return all(c % 2 == 0 for c in Counter(x % k for x in arr).values())

print(can_pair_diff([1, 4, 7, 10], 3), can_pair_diff([1, 2, 3, 4], 3))

True False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | Thử mọi cách ghép | hàm mũ | `O(n)` | Kiểm chứng, `n ≤ 12` |
| 2. Ghép tham lam | Hàng chờ theo số dư, ghép ngay khi có đối tác | `O(n)` | `O(min(n,k))` | Một lượt, xử lý được dòng dữ liệu |
| 3. Đếm số dư | `cnt[0]`, `cnt[k/2]` chẵn; `cnt[r] = cnt[k−r]` | `O(n + k)` | `O(k)` | Bản nộp chuẩn |
| 4. Sort + hai con trỏ | Nhỏ nhất ghép lớn nhất, tổng phải bằng `k` | `O(n log n)` | `O(n)` | `k` cực lớn, không lập mảng `cnt` |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().canArrange([-1, 1, -2, 2, -3, 3, -4, 4], 3)` trả gì? Ghi mảng số dư.

### Bài 2 (tính tay)
Với `arr = [9, 7, 5, 3], k = 6`: lập `cnt` và kiểm tra từng điều kiện.

### Bài 3 (điền TODO)
Hoàn thiện `make_pairs(arr, k)`: trả về **một danh sách cặp** cụ thể (các bộ `(a, b)`) nếu ghép được, ngược lại `None`.

### Bài 4 (tự viết lại)
Viết `count_pairs(arr, k)`: đếm số cặp chỉ số `i < j` với `(arr[i] + arr[j]) % k == 0` trong `O(n + k)`.

### Bài 5 (mini challenge)
Viết `min_remove(arr, k)`: số phần tử **ít nhất** cần bỏ đi để phần còn lại chia được thành các cặp có tổng chia hết cho `k` (phần còn lại có thể rỗng).

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def make_pairs(arr, k):
    # TODO: gom phần tử theo số dư (dict r -> list), rồi ghép nhóm r với nhóm (k - r) % k
    pass

# Bài 4 — TODO
def count_pairs(arr, k):
    # TODO: duyệt x, cộng cnt[(k - x % k) % k] vào kết quả, rồi cnt[x % k] += 1
    pass

# Bài 5 — TODO
def min_remove(arr, k):
    # TODO: nhóm 0, nhóm k/2: bỏ cnt % 2; cặp (r, k-r): bỏ |cnt[r] - cnt[k-r]|
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Số dư: `2,1,1,2,0,0,2,1`. (2) `cnt = [2, 3, 3]`. (3) `cnt[0]` chẵn, `cnt[1] = cnt[2]` ⇒ `True`.

**Bài 2.** (1) Số dư `3, 1, 5, 3`. (2) `cnt[0] = 0` chẵn; `cnt[1] = cnt[5] = 1`; `cnt[2] = cnt[4] = 0`. (3) `cnt[3] = 2` chẵn ⇒ `True` (cặp `(9,3)`, `(7,5)`).

**Bài 3.** (1) `groups = defaultdict(list)`. (2) Với `r = 0` và `r = k/2`: ghép từng đôi trong nhóm. (3) Với `r < k − r`: kiểm tra độ dài bằng nhau rồi `zip`.

**Bài 4.** (1) Đếm "online": trước khi thêm `x`, số phần tử trước đó ghép được với `x` là `cnt[(k − x%k) % k]`. (2) Cộng dồn. (3) Kiểm tra bằng hai vòng lặp với `n` nhỏ.

**Bài 5.** (1) Mỗi cặp nhóm `(r, k−r)` độc lập. (2) Thừa ra bao nhiêu thì bỏ bấy nhiêu: `|cnt[r] − cnt[k−r]|`. (3) Nhóm tự đối tác thừa tối đa 1.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
arr1 = [-1, 1, -2, 2, -3, 3, -4, 4]
print([x % 3 for x in arr1], Solution().canArrange(arr1, 3))
assert [x % 3 for x in arr1] == [2, 1, 1, 2, 0, 0, 2, 1] and Solution().canArrange(arr1, 3)

# Bài 2
cnt2 = [0] * 6
for x in [9, 7, 5, 3]:
    cnt2[x % 6] += 1
print(cnt2)
assert cnt2 == [0, 1, 0, 2, 0, 1] and Solution().canArrange([9, 7, 5, 3], 6)

[2, 1, 1, 2, 0, 0, 2, 1] True
[0, 1, 0, 2, 0, 1]


In [25]:
# Bài 3
from collections import defaultdict
def make_pairs_sol(arr, k):
    groups = defaultdict(list)
    for x in arr:
        groups[x % k].append(x)
    pairs = []
    for r in range(0, k // 2 + 1):
        s = (k - r) % k
        if r == s:
            g = groups[r]
            if len(g) % 2:
                return None
            pairs += [(g[i], g[i + 1]) for i in range(0, len(g), 2)]
        else:
            if len(groups[r]) != len(groups[s]):
                return None
            pairs += list(zip(groups[r], groups[s]))
    return pairs

# Bài 4
def count_pairs_sol(arr, k):
    cnt = [0] * k
    total = 0
    for x in arr:
        r = x % k
        total += cnt[(k - r) % k]
        cnt[r] += 1
    return total

rng = random.Random(3)
for _ in range(1500):
    arr, k = random_case(rng)
    ps = make_pairs_sol(arr, k)
    if Solution().canArrange(arr, k):
        assert ps is not None and len(ps) == len(arr) // 2
        assert sorted(v for p in ps for v in p) == sorted(arr)
        assert all((a + b) % k == 0 for a, b in ps)
    else:
        assert ps is None
    brute_cnt = sum((arr[i] + arr[j]) % k == 0 for i in range(len(arr)) for j in range(i + 1, len(arr)))
    assert count_pairs_sol(arr, k) == brute_cnt
print("Bài 3, 4: OK |", make_pairs_sol([1, 2, 3, 4, 5, 10, 6, 7, 8, 9], 5))

Bài 3, 4: OK | [(5, 10), (1, 4), (6, 9), (2, 3), (7, 8)]


In [26]:
# Bài 5
from itertools import combinations
def min_remove_sol(arr, k):
    cnt = [0] * k
    for x in arr:
        cnt[x % k] += 1
    res = cnt[0] % 2
    for r in range(1, k // 2 + 1):
        if r == k - r:
            res += cnt[r] % 2
        else:
            res += abs(cnt[r] - cnt[k - r])
    return res

def min_remove_brute(arr, k):
    n = len(arr)
    for t in range(n + 1):
        for rem in combinations(range(n), t):
            s = set(rem)
            rest = [arr[i] for i in range(n) if i not in s]
            if len(rest) % 2 == 0 and can_arrange_brute(rest, k):
                return t
    return n

print(min_remove_sol([1, 2, 3, 4, 5, 6], 10), min_remove_sol([1, 1, 1, 3], 3))
for _ in range(400):
    k = rng.randint(1, 6)
    arr = [rng.randint(-10, 10) for _ in range(rng.randint(0, 8))]
    assert min_remove_sol(arr, k) == min_remove_brute(arr, k), (arr, k)
print("Đáp án Bài 1–5: mọi assert đều qua")

4 4
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "ghép cặp tổng chia hết cho k":**

```python
cnt = [0] * k
for x in arr:
    cnt[x % k] += 1                 # C++/Java: ((x % k) + k) % k
if cnt[0] % 2: return False
for r in range(1, k // 2 + 1):
    if r == k - r:
        if cnt[r] % 2: return False # nhóm k/2
    elif cnt[r] != cnt[k - r]:
        return False
return True
```

**Checklist trước khi submit:**
- [ ] Số dư của số âm đã chuẩn hoá về `[0, k)`.
- [ ] Nhóm `0` chẵn; nhóm `k/2` (khi `k` chẵn) chẵn.
- [ ] Không truy cập `cnt[k]` (dùng `(k − r) % k` hoặc vòng từ 1).
- [ ] Không chỉ kiểm tra tổng cả mảng.
- [ ] Test: `k = 1`, `k = 2`, số âm, `[2,1,3,4], k=4`.
- [ ] Đúng chữ ký: `canArrange(self, arr: list[int], k: int) -> bool`.